# 03 - Run the external agent with logs and traces

This notebook runs a real external agent using a model deployment in the Foundry project. Each invocation creates an `invoke_agent` span and correlated Python log records in Application Insights.

Restart the kernel before rerunning the telemetry initialization cells.

In [ ]:
import logging
import os
import time
from pathlib import Path

from dotenv import load_dotenv

if not Path('.env').exists():
    raise FileNotFoundError('Copy .env.example to .env and fill in the required values.')

load_dotenv('.env')

required_variables = (
    'FOUNDRY_PROJECT_ENDPOINT',
    'APPLICATIONINSIGHTS_CONNECTION_STRING',
    'AGENT_NAME',
    'OTEL_AGENT_ID',
    'MODEL_DEPLOYMENT_NAME',
)
missing = [name for name in required_variables if not os.getenv(name)]
if missing:
    raise ValueError(f'Missing required environment variables: {missing}')

os.environ.setdefault('AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING', 'true')
os.environ.setdefault('OTEL_SEMCONV_STABILITY_OPT_IN', 'gen_ai_latest_experimental')

log_message_content = os.getenv('LOG_MESSAGE_CONTENT', 'false').lower() == 'true'
print(f"Agent: {os.environ['AGENT_NAME']}")
print(f"Model deployment: {os.environ['MODEL_DEPLOYMENT_NAME']}")
print(f'Log prompt and response content: {log_message_content}')

In [ ]:
from microsoft.opentelemetry import use_microsoft_opentelemetry

use_microsoft_opentelemetry(
    enable_azure_monitor=True,
    azure_monitor_connection_string=os.environ['APPLICATIONINSIGHTS_CONNECTION_STRING'],
    sampling_ratio=1.0,
)

logger = logging.getLogger('external-agent')
logger.setLevel(logging.INFO)

if not any(getattr(handler, '_external_agent_console', False) for handler in logger.handlers):
    console_handler = logging.StreamHandler()
    console_handler._external_agent_console = True
    console_handler.setFormatter(
        logging.Formatter('%(asctime)s %(levelname)s %(name)s - %(message)s')
    )
    logger.addHandler(console_handler)

logger.info('External-agent telemetry initialized')

In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=os.environ['FOUNDRY_PROJECT_ENDPOINT'],
    credential=credential,
    allow_preview=True,
)
openai_client = project.get_openai_client()
tracer = trace.get_tracer('external-agent-runtime')

available_deployments = {deployment.name for deployment in project.deployments.list()}
model_deployment = os.environ['MODEL_DEPLOYMENT_NAME']
if model_deployment not in available_deployments:
    raise ValueError(
        f'Model deployment {model_deployment!r} was not found. '
        f'Available deployments: {sorted(available_deployments)}'
    )

print('Foundry client and model deployment are ready.')

In [ ]:
def run_agent(prompt: str) -> str:
    if not prompt.strip():
        raise ValueError('The prompt cannot be empty.')

    started = time.perf_counter()

    with tracer.start_as_current_span('invoke_agent') as span:
        span.set_attribute('gen_ai.operation.name', 'invoke_agent')
        span.set_attribute('gen_ai.agent.id', os.environ['OTEL_AGENT_ID'])
        span.set_attribute('gen_ai.agent.name', os.environ['AGENT_NAME'])
        span.set_attribute('gen_ai.request.model', model_deployment)
        span.set_attribute('external_agent.prompt_length', len(prompt))

        span_context = span.get_span_context()
        trace_id = format(span_context.trace_id, '032x')
        span_id = format(span_context.span_id, '016x')

        logger.info(
            'Agent invocation started trace_id=%s span_id=%s prompt_length=%d',
            trace_id,
            span_id,
            len(prompt),
        )
        if log_message_content:
            logger.info('Agent prompt: %s', prompt)

        try:
            response = openai_client.responses.create(
                model=model_deployment,
                instructions=(
                    'You are a concise, helpful external agent registered in '
                    'Microsoft Foundry for observability and evaluation.'
                ),
                input=prompt,
            )
            answer = response.output_text
            duration_ms = (time.perf_counter() - started) * 1000

            span.set_attribute('gen_ai.response.id', response.id)
            span.set_attribute('gen_ai.response.model', response.model)
            span.set_attribute('external_agent.response_length', len(answer))
            span.set_attribute('external_agent.duration_ms', duration_ms)

            usage = response.usage
            if usage is not None:
                span.set_attribute('gen_ai.usage.input_tokens', usage.input_tokens)
                span.set_attribute('gen_ai.usage.output_tokens', usage.output_tokens)
                span.set_attribute('gen_ai.usage.total_tokens', usage.total_tokens)

            logger.info(
                'Agent invocation completed trace_id=%s duration_ms=%.1f '
                'input_tokens=%s output_tokens=%s',
                trace_id,
                duration_ms,
                getattr(usage, 'input_tokens', None),
                getattr(usage, 'output_tokens', None),
            )
            if log_message_content:
                logger.info('Agent response: %s', answer)

            return answer
        except Exception as error:
            duration_ms = (time.perf_counter() - started) * 1000
            span.record_exception(error)
            span.set_status(Status(StatusCode.ERROR, str(error)))
            span.set_attribute('external_agent.duration_ms', duration_ms)
            logger.exception(
                'Agent invocation failed trace_id=%s duration_ms=%.1f',
                trace_id,
                duration_ms,
            )
            raise
        finally:
            provider = trace.get_tracer_provider()
            if hasattr(provider, 'force_flush'):
                provider.force_flush(timeout_millis=30000)

## Invoke the agent

Rerun this cell for each new interaction. The answer appears in the notebook, while the invocation span and correlated logs are exported to Application Insights.

In [ ]:
prompt = input('You: ')
answer = run_agent(prompt)
print(f'\nAgent: {answer}')

## View new telemetry

- **Foundry:** open the project, then **Build → Agents → my-external-agent → Traces**.
- **Application Insights:** use **Transaction search** for correlated requests, dependencies, traces, and exceptions.
- Allow approximately 2-5 minutes for ingestion and portal display.

Prompt and response text are not logged by default. Set `LOG_MESSAGE_CONTENT=true` in `.env` only when your data-handling policy permits it.